In [1]:
using CSV, DataFrames, Statistics, Printf, StatsBase, Plots

include("functions.jl")

data_dir  = "output"
burnin    = 500_000
thin      = 10
chain_ids = 1:3
outfile   = joinpath(data_dir, "rhat_summary.csv")

filename_for_chain(c) = joinpath(data_dir, "samples_chain_$(c).csv")

function load_chain_df(path::String; burnin::Int, thin::Int)
    df = CSV.read(path, DataFrame)
    idx = (burnin + 1):thin:nrow(df)
    return df[idx, :]
end

dfs = DataFrame[]
for c in chain_ids
    fpath = filename_for_chain(c)
    push!(dfs, load_chain_df(fpath; burnin=burnin, thin=thin))
end

n_keep = minimum(nrow.(dfs))
dfs = [df[1:n_keep, :] for df in dfs]

params = String.(names(dfs[1]))
params = [p for p in params if eltype(dfs[1][!, p]) <: Real]

rhat_vals = Float64[]
rhat_names = String[]

for p in params
    mat = Array{Float64}(undef, length(chain_ids), n_keep)
    for (i, df) in enumerate(dfs)
        mat[i, :] = Float64.(df[!, p])
    end
    push!(rhat_names, p)
    push!(rhat_vals, rhat_gelman_rubin(mat))
end

outdf = DataFrame(param = rhat_names, Rhat = rhat_vals)
CSV.write(outfile, outdf)

println("Wrote Rhat summary to: $outfile")


Wrote Rhat summary to: output/rhat_summary.csv


In [ ]:
using DelimitedFiles
using Statistics
using StatsBase
using Random
using Printf


const OUTPUT_DIR        = "output"
const N_CHAINS          = 3
const BURN_IN_SAMPLES   = 500_000    
const THIN_SAMPLES      = 10    

const BURN_IN_LOGLIK    = 0
const THIN_LOGLIK       = 1

param_header = ["beta","alpha","gamma","lambda_R"]

const N_DRAWS = 1000

function read_csv_matrix(path::String)
    data, hdr = DelimitedFiles.readdlm(path, ',', header=true)
    M = Matrix{Float64}(data)
    header = hdr === nothing ? nothing : vec(collect(hdr))
    return M, header
end

function load_samples_for_chain(c::Int, outdir::String)
    path = joinpath(outdir, "samples_chain_$(c).csv")
    if !isfile(path)
        @warn "Missing samples file: $path"; return Array{Float64}(undef, 0, 0)
    end
    M, _ = read_csv_matrix(path)
    if size(M,1) <= BURN_IN_SAMPLES
        @warn "File $path has only $(size(M,1)) rows (<= burn-in)."
        return Array{Float64}(undef, 0, 0)
    end
    idxs = collect(BURN_IN_SAMPLES+1:THIN_SAMPLES:size(M,1))
    return M[idxs, :]
end

function load_loglik_for_chain(c::Int, outdir::String)
    path = joinpath(outdir, "loglik_chain_$(c).csv")
    if !isfile(path)
        @warn "Missing loglik file: $path"; return Array{Float64}(undef, 0, 0)
    end
    M, _ = read_csv_matrix(path)
    if BURN_IN_LOGLIK > 0 || THIN_LOGLIK > 1
        idxs = collect(BURN_IN_LOGLIK+1:THIN_LOGLIK:size(M,1))
        M = M[idxs, :]
    end
    return M
end

function logmeanexp_columnwise(L::AbstractMatrix{<:Real})
    S, T = size(L)
    out = Vector{Float64}(undef, T)
    for j in 1:T
        col = L[:, j]
        m = maximum(col)
        out[j] = m + log(sum(exp.(col .- m)) / S)
    end
    return out
end

function compute_waic_from_list(all_loglik_aug_vecs::Vector{Matrix{Float64}})
    combined_logliks = vcat(all_loglik_aug_vecs...)
    if any(isinf, combined_logliks)
        @warn "Encountered infinite log-likelihoods. WAIC will be -Inf."
        return -Inf
    end
    lppd = sum(logmeanexp_columnwise(combined_logliks))
    pwaic = sum(var(combined_logliks, dims=1))
    return -2 * lppd + 2 * pwaic
end

function summarize_posterior(samples::AbstractMatrix{<:Real}, header_cont::Vector{String})
    summary_dict = Dict{String, Any}()
    for (i, p_name) in enumerate(header_cont)
        param_samples = samples[:, i]
        if p_name == "k_max"
            k = mode(round.(Int, param_samples))
            freq = count(==(k), round.(Int, param_samples)) / length(param_samples)
            summary_dict[p_name] = Dict("mode" => k, "frequency" => freq)
        else
            med = median(param_samples)
            q = quantile(param_samples, [0.025, 0.975])
            summary_dict[p_name] = Dict("median" => med, "95%CI_low" => q[1], "95%CI_high" => q[2])
        end
    end
    return summary_dict
end

function write_csv(path::String, header::Vector{String}, M::AbstractMatrix)
    open(path, "w") do io
        println(io, join(header, ","))
        DelimitedFiles.writedlm(io, M, ',')
    end
end

function write_summary_csv(path::String, summary_dict::Dict{String,Any})
    open(path, "w") do io
        println(io, "Parameter,Median_or_Mode,CI_Low_or_Frequency,CI_High")
        for (param, vals) in summary_dict
            if haskey(vals, "mode")
                println(io, "$param,$(vals["mode"]),$(vals["frequency"]),")
            else
                println(io, "$param,$(vals["median"]),$(vals["95%CI_low"]),$(vals["95%CI_high"])")
            end
        end
    end
end

Random.seed!(2025)

samples_all = Matrix{Float64}[]
for c in 1:N_CHAINS
    push!(samples_all, load_samples_for_chain(c, OUTPUT_DIR))
end
samples_bt = vcat(samples_all...)
if size(samples_bt,1) == 0
    error("No samples found")
end

posterior_summary = summarize_posterior(samples_bt, param_header)

ll_list = Matrix{Float64}[]
for c in 1:N_CHAINS
    M = load_loglik_for_chain(c, OUTPUT_DIR)
    if size(M,1) > 0
        push!(ll_list, M)
    end
end
waic_val = isempty(ll_list) ? NaN : compute_waic_from_list(ll_list)

n_rows = size(samples_bt, 1)
replace_flag = n_rows < N_DRAWS
draw_indices = sample(1:n_rows, N_DRAWS; replace=replace_flag)
draw_params  = samples_bt[draw_indices, :]

draws_outfile = joinpath(OUTPUT_DIR, @sprintf("posterior_draws.csv"))
hdr = vcat(["row_index"], param_header)
Mout = hcat(Float64.(draw_indices), draw_params)
write_csv(draws_outfile, hdr, Mout)

write_summary_csv(joinpath(OUTPUT_DIR, "posterior_summary.csv"), posterior_summary)

open(joinpath(OUTPUT_DIR, "waic.csv"), "w") do io
    println(io, "WAIC")
    println(io, waic_val)
end

@info "Done. Outputs written in $(OUTPUT_DIR)"


[ Info: Done. Outputs written in output


In [ ]:
using Random, Distributions, Statistics, Printf, DelimitedFiles
using Plots
Random.seed!(2025) 
# ---------- helpers ----------
function clamp01(x)
    x < 0 ? 0.0 : (x > 1 ? 1.0 : x)
end

# a_t = 1 - (1 - ψ_t)^(1/α)
function baseline_alarm(psi, alpha)
    psi = clamp01(psi)
    alpha <= 0 && error("alpha must be > 0")
    1 - (1 - psi)^(1/alpha)
end

psi_memoryless(Istar_hist, N) = isempty(Istar_hist) ? 0.0 : Istar_hist[end] / N

function psi_sliding(Istar_hist, N, k_max)
    L = length(Istar_hist)
    L == 0 && return 0.0
    m = min(k_max, L)
    mean(@view(Istar_hist[(L - m + 1):L])) / N
end

function psi_powerlaw(Istar_hist, N, lambda_P)
    L = length(Istar_hist)
    L == 0 && return 0.0
    num  = 0.0
    @inbounds for j in 1:L
        w = j^(-lambda_P)
        num  += w * Istar_hist[end - j + 1]
    end
    num / N  
end

function psi_exponential(Istar_hist, N, lambda_E)
    L = length(Istar_hist)
    L == 0 && return 0.0
    num  = 0.0
    @inbounds for j in 0:(L - 1)
        w = exp(-lambda_E * j)
        num  += w * Istar_hist[end - j]
    end
    num / N   
end

function psi_reciprocal(Istar_hist, N, lambda_R)
    lambda_R <= 0 && error("lambda_R must be > 0")
    L = length(Istar_hist)
    L == 0 && return 0.0
    num = 0.0
    @inbounds for j in 0:(L - 1)
        w = 1.0 / (1.0 + lambda_R * j)   
        num += w * Istar_hist[end - j]
    end
    num / N
end

# ---------- simulate one epidemic ----------
function simulate_epidemic(N, I0, tau, beta, alpha, rateI;
                           mechanism::Symbol = :memoryless,
                           k_max::Union{Nothing,Int}=nothing,
                           lambda_P::Union{Nothing,Float64}=nothing,
                           lambda_E::Union{Nothing,Float64}=nothing,
                           lambda_R::Union{Nothing,Float64}=nothing)

    S      = zeros(Int, tau + 1)
    I      = zeros(Int, tau + 1)
    Istar  = zeros(Int, tau)
    Rstar  = zeros(Int, tau)
    psi    = zeros(Float64, tau)
    alarm  = zeros(Float64, tau)
    probSI = zeros(Float64, tau)
    probIR = 1 - exp(-rateI)

    S[1] = 2_701_767 - 3 
    I[1] = I0

    # t = 1
    psi[1]   = 0.0
    alarm[1] = baseline_alarm(psi[1], alpha)
    pSI      = clamp01(1 - exp(-beta * (1 - alarm[1]) * (I[1] / N)))
    probSI[1] = pSI
    Istar[1] = rand(Binomial(S[1], pSI))
    Rstar[1] = rand(Binomial(I[1], probIR))
    S[2]     = S[1] - Istar[1]
    I[2]     = I[1] + Istar[1] - Rstar[1]

    # t = 2..tau
    for t in 2:tau
        hist = @view Istar[1:(t-1)]
        ψt = if mechanism === :memoryless
            psi_memoryless(hist, N)
        elseif mechanism === :sliding
            isnothing(k_max) && error("k_max must be provided for :sliding")
            psi_sliding(hist, N, k_max)
        elseif mechanism === :powerlaw
            isnothing(lambda_P) && error("lambda_P must be provided for :powerlaw")
            psi_powerlaw(hist, N, lambda_P)
        elseif mechanism === :exponential
            isnothing(lambda_E) && error("lambda_E must be provided for :exponential")
            psi_exponential(hist, N, lambda_E)
        elseif mechanism === :reciprocal
            isnothing(lambda_R) && error("lambda_R must be provided for :reciprocal")
            psi_reciprocal(hist, N, lambda_R)
        else
            error("Unknown mechanism: $mechanism")
        end
        psi[t]   = ψt
        alarm[t] = baseline_alarm(ψt, alpha)
        pSI      = clamp01(1 - exp(-beta * (1 - alarm[t]) * (I[t] / N)))
        probSI[t] = pSI

        Istar[t] = rand(Binomial(S[t], pSI))
        Rstar[t] = rand(Binomial(I[t], probIR))

        S[t+1] = S[t] - Istar[t]
        I[t+1] = I[t] + Istar[t] - Rstar[t]
    end

    return Dict(
        :Istar => Istar,
        :Rstar => Rstar,
        :S     => S,
        :I     => I,
        :psi   => psi,
        :alarm => alarm,
        :probSI => probSI,
        :probIR => probIR
    )
end

# ---------- write CSV ----------
function write_matrix_csv(path::String, header::Vector{String}, M::AbstractMatrix)
    open(path, "w") do io
        println(io, join(header, ","))
        for i in 1:size(M,1)
            println(io, join(M[i, :], ","))
        end
    end
end

OUTPUT_DIR = "output"
TAU = 59
N_pop, I0 = 2_701_767 , 3 
MECHANISM = :reciprocal
params = ["beta","alpha","gamma","k_max","lambda_P","lambda_E","lambda_R"]

function read_draws(path)
    data,hdr = readdlm(path,',',header=true)
    Matrix{Float64}(data),vec(hdr)
end

function idxmap(header,names)
    H=Dict(n=>i for (i,n) in enumerate(header))
    Dict(n=>get(H,n,nothing) for n in names)
end

function summarize(M)
    τ=size(M,2); med=Float64[]; lo=Float64[]; hi=Float64[]
    for t=1:τ
        col=M[:,t]; push!(med,median(col))
        q=quantile(col,[0.025,0.975]); push!(lo,q[1]); push!(hi,q[2])
    end
    med,lo,hi
end

function write_csv(path,hdr,rows)
    open(path,"w") do io
        println(io,join(hdr,",")); foreach(r->println(io,join(r,",")),rows)
    end
end

function avg_series(list)
    τ=length(list[1]); [mean([s[t] for s in list]) for t=1:τ]
end

istarM=[]; istarL=[]; istarH=[]; alarmM=[]; alarmL=[]; alarmH=[]
M,hdr=read_draws("$(OUTPUT_DIR)/posterior_draws.csv")
col=idxmap(hdr,params); n=size(M,1)
istar=zeros(n,TAU); alarm=zeros(n,TAU)
nsim_per_draw = 100 

istar = zeros(n, TAU)
alarm = zeros(n, TAU)

for k in 1:n
    β = M[k, col["beta"]]
    α = M[k, col["alpha"]]
    γ = M[k, col["gamma"]]
    kmax = col["k_max"] === nothing ? nothing : round(Int, M[k, col["k_max"]])
    λP = col["lambda_P"] === nothing ? nothing : M[k, col["lambda_P"]]
    λE = col["lambda_E"] === nothing ? nothing : M[k, col["lambda_E"]]
    λR = col["lambda_R"] === nothing ? nothing : M[k, col["lambda_R"]]

    Iblock = Array{Float64}(undef, nsim_per_draw, TAU)
    Ablock = Array{Float64}(undef, nsim_per_draw, TAU)

    for s in 1:nsim_per_draw
        sim = simulate_epidemic(N_pop, I0, TAU, β, α, γ;
                                mechanism = :reciprocal,
                                k_max = kmax, lambda_P = λP, lambda_E = λE, lambda_R = λR)
        Iblock[s, :] = sim[:Istar]
        Ablock[s, :] = sim[:alarm]
    end

    istar[k, :] = vec(mapslices(median, Iblock; dims = 1))
    alarm[k, :] = vec(mapslices(median, Ablock; dims = 1))
end

    m,l,h=summarize(istar); write_csv("$(OUTPUT_DIR)/Istar_stats.csv",
        ["Day","median","ci_low","ci_high"],[[t,m[t],l[t],h[t]] for t=1:TAU])
    m,l,h=summarize(alarm); write_csv("$(OUTPUT_DIR)/alarm_stats.csv",
        ["Day","median","ci_low","ci_high"],[[t,m[t],l[t],h[t]] for t=1:TAU])
    push!(istarM,m);push!(istarL,l);push!(istarH,h)
    push!(alarmM,m);push!(alarmL,l);push!(alarmH,h)


1-element Vector{Any}:
 [0.0, 0.0007615715001399848, 0.001532402971605387, 0.002466690826279978, 0.003695988425482961, 0.005315162236718389, 0.007516656342853127, 0.010225990701862872, 0.01395452291239189, 0.018816340916361307  …  0.6424151874983561, 0.6414911213955623, 0.6398681544509935, 0.6384468996817467, 0.6365400937971548, 0.6350238094322406, 0.6324682153202618, 0.6311096203769823, 0.6294751249832173, 0.627451008250543]

In [4]:
Istar_obs = [
2, 6, 11, 14, 17, 23, 31, 38, 43, 46, 74, 91, 119, 138, 193, 255, 257,
324, 372, 412, 422, 407, 411, 450, 408, 394, 371, 416, 425, 388, 387,
369, 386, 365, 328, 314, 335, 298, 323, 300, 280, 285, 273, 254, 253,
211, 209, 232, 203, 217, 199, 206, 217, 182, 173, 176, 154, 166, 157
]
istar_mat = readdlm(joinpath(OUTPUT_DIR, "Istar_stats.csv"), ',', header=true)[1]
T = min(size(istar_mat, 1), length(Istar_obs))
out = hcat(istar_mat[1:T, :], Istar_obs[1:T])
open(joinpath(OUTPUT_DIR, "Istar_stats_with_truth.csv"), "w") do io
    println(io, "Day,median,ci_low,ci_high,truth")
    writedlm(io, out, ',')
end

In [2]:
using Random, Statistics, Printf, DelimitedFiles

OUTPUT_DIR = "output"
TAU        = 59
N_pop      = 2_701_767
I0         = 3
nsim_per_draw_Rt = 100
Random.seed!(2025)

params = ["beta","alpha","gamma","k_max","lambda_P","lambda_E","lambda_R"]
function read_draws(path)
    data,hdr = readdlm(path, ',', header=true)
    Matrix{Float64}(data), vec(hdr)
end
M, hdr = read_draws("$(OUTPUT_DIR)/posterior_draws.csv")
H = Dict(n=>i for (i,n) in enumerate(hdr))
col = Dict(n=>get(H,n,nothing) for n in params)
n = size(M,1)

function compute_Rt_from_sim(beta::Real, alarm::AbstractVector{<:Real},
                             rateI::Real, N::Real, S::AbstractVector{<:Real};
                             maxInf::Int=14)
    T = length(alarm)
    Rt = fill(NaN, T)
    pi_IR = 1 - exp(-rateI)
    w = (1 - pi_IR) .^ (0:maxInf-1)
    @inbounds for t in 1:(T - maxInf)
        seg = t:(t + maxInf - 1)
        pi_SI = 1 .- exp.(- (beta .* (1 .- alarm[seg])) ./ N)
        Rt[t] = S[t] * sum(pi_SI .* w)
    end
    return Rt
end

function summarize_skipna(M::AbstractMatrix)
    τ=size(M,2); med=Float64[]; lo=Float64[]; hi=Float64[]
    for t=1:τ
        col = M[:,t]; col = col[.!isnan.(col)]
        if isempty(col)
            push!(med, NaN); push!(lo, NaN); push!(hi, NaN)
        else
            push!(med, median(col))
            q = quantile(col, [0.025, 0.975])
            push!(lo, q[1]); push!(hi, q[2])
        end
    end
    med, lo, hi
end

function write_csv(path,hdr,rows)
    open(path,"w") do io
        println(io,join(hdr,",")); foreach(r->println(io,join(r,",")),rows)
    end
end

Rt_mat = fill(NaN, n, TAU)

for k in 1:n
    β = M[k, col["beta"]]
    α = M[k, col["alpha"]]
    γ = M[k, col["gamma"]]
    kmax = col["k_max"] === nothing ? nothing : round(Int, M[k, col["k_max"]])
    λP = col["lambda_P"] === nothing ? nothing : M[k, col["lambda_P"]]
    λE = col["lambda_E"] === nothing ? nothing : M[k, col["lambda_E"]]
    λR = col["lambda_R"] === nothing ? nothing : M[k, col["lambda_R"]]

    Rblock = Array{Float64}(undef, nsim_per_draw_Rt, TAU)
    for s in 1:nsim_per_draw_Rt
        sim = simulate_epidemic(N_pop, I0, TAU, β, α, γ;
                                mechanism = :reciprocal,
                                k_max = kmax, lambda_P = λP,
                                lambda_E = λE, lambda_R = λR)
        Rblock[s, :] = compute_Rt_from_sim(β, sim[:alarm], γ, N_pop, sim[:S][1:end-1])
    end
    Rt_mat[k, :] = vec(mapslices(median, Rblock; dims = 1))
end

mRt, lRt, hRt = summarize_skipna(Rt_mat)

rows = [[t, mRt[t], lRt[t], hRt[t]] for t in 1:TAU if !(isnan(mRt[t]) || isnan(lRt[t]) || isnan(hRt[t]))]

write_csv("$(OUTPUT_DIR)/Rt_stats.csv",
    ["Day","median","ci_low","ci_high"],
    rows)

println("Rt_stats.csv written to $(OUTPUT_DIR)")


Rt_stats.csv written to output
